# AIML Lab 5 - Clinical Text Processing with Chunking and NER

**Aim:** implement an NLP pipeline that performs chunking and Named Entity
Recognition on unstructured clinical text to identify diseases, symptoms,
medications, investigations, dosage and anatomical locations.

**Dataset:** 12 short synthetic clinical notes in `clinical_notes.csv`. No real
patient information is used, and no names, phone numbers, addresses or hospital
IDs appear in the notes.

**Steps covered:** load and tokenize -> chunking -> NER -> combined extraction
pipeline -> evaluation -> industry use case.

## Part A - Load and Preprocess Clinical Text

### A1. Import the required libraries

In [1]:
import pandas as pd
import spacy
from spacy import displacy
from spacy.tokens import Span

pd.set_option("display.max_colwidth", 60)
print("spaCy version:", spacy.__version__)

spaCy version: 3.8.16


### A2. Load the clinical text dataset

In [2]:
notes_df = pd.read_csv("clinical_notes.csv")
notes_df.head()

,note_id,note_text
0,1,"A 56-year-old male presented with fever, productive coug..."
1,2,A patient with type 2 diabetes mellitus reported increas...
2,3,The patient developed severe chest pain radiating to the...
3,4,"A young female presented with headache, photophobia and ..."
4,5,Patient complained of burning micturition and lower abdo...


### A3. Number of clinical records

In [3]:
print("Number of Clinical Records:", len(notes_df))

Number of Clinical Records: 12


### A4. Select one clinical note for processing

In [4]:
nlp = spacy.load("en_core_web_sm")

note = notes_df.loc[0, "note_text"]

print("Clinical Note:")
print(note)

Clinical Note:
A 56-year-old male presented with fever, productive cough and chest pain for 3 days. Chest X-ray showed right lower-lobe infiltrates suggestive of pneumonia. The patient was prescribed Azithromycin 500 mg once daily for 5 days and Paracetamol 650 mg as required.


### A5 and A6. Tokenization

In [5]:
doc = nlp(note)
tokens = [token.text for token in doc]

print("Number of Tokens:", len(tokens))
print()
print("Tokens:")
print(" | ".join(tokens))

Number of Tokens: 53

Tokens:
A | 56 | - | year | - | old | male | presented | with | fever | , | productive | cough | and | chest | pain | for | 3 | days | . | Chest | X | - | ray | showed | right | lower | - | lobe | infiltrates | suggestive | of | pneumonia | . | The | patient | was | prescribed | Azithromycin | 500 | mg | once | daily | for | 5 | days | and | Paracetamol | 650 | mg | as | required | .


A shorter example, in the format used in the lab sheet.

In [6]:
short_note = "Patient presented with fever and persistent cough."
short_doc = nlp(short_note)

print("Clinical Note:")
print(short_note)
print()
print("Tokens:")
print(" | ".join(token.text for token in short_doc))

Clinical Note:
Patient presented with fever and persistent cough.

Tokens:
Patient | presented | with | fever | and | persistent | cough | .


## Part B - Implement Chunking

Noun phrase chunking groups related tokens into one phrase, so that
"severe chest pain" is handled as a single clinical concept instead of three
separate words.

In [7]:
chunk_example = nlp("The patient developed severe chest pain and persistent cough.")

rows = [{"Chunk": chunk.text, "Type": "Noun Phrase"} for chunk in chunk_example.noun_chunks]
pd.DataFrame(rows)

,Chunk,Type
0,The patient,Noun Phrase
1,severe chest pain,Noun Phrase
2,persistent cough,Noun Phrase


### Chunks in the selected clinical note, with dependency information

In [8]:
chunk_rows = []
for chunk in doc.noun_chunks:
    chunk_rows.append({
        "Chunk": chunk.text,
        "Root": chunk.root.text,
        "Root POS": chunk.root.pos_,
        "Dependency": chunk.root.dep_,
        "Head": chunk.root.head.text,
    })

chunks_df = pd.DataFrame(chunk_rows)
print("Number of noun phrases found:", len(chunks_df))
chunks_df

Number of noun phrases found: 15


,Chunk,Root,Root POS,Dependency,Head
0,A 56-year-old male,male,NOUN,ROOT,male
1,fever,fever,NOUN,pobj,with
2,productive cough,cough,NOUN,conj,fever
3,chest pain,pain,NOUN,conj,cough
4,3 days,days,NOUN,pobj,for
5,Chest X,X,NOUN,nsubj,showed
6,-,-,NOUN,nsubj,showed
7,ray,ray,NOUN,nsubj,showed
8,lower-lobe,lobe,NOUN,nsubj,infiltrates
9,suggestive,suggestive,NOUN,dobj,infiltrates


### Clinically meaningful phrases

Not every noun phrase is useful. Phrases such as "the patient" carry no clinical
information, so a simple filter keeps only the multi-word phrases and drops the
ones made up entirely of stopwords and pronouns.

In [9]:
STOP_CHUNKS = {"the patient", "a patient", "patient", "it", "he", "she", "they", "days"}


def is_meaningful(chunk):
    text = chunk.text.lower().strip()
    if text in STOP_CHUNKS:
        return False
    return any(not token.is_stop and not token.is_punct for token in chunk)


for chunk in doc.noun_chunks:
    if is_meaningful(chunk):
        print("-", chunk.text)

- A 56-year-old male
- fever
- productive cough
- chest pain
- 3 days
- Chest X
- ray
- lower-lobe
- suggestive
- pneumonia
- 500 mg
- 5 days
- Paracetamol 650 mg


## Part C - Implement Named Entity Recognition

### C1. NER using the general pretrained model

First the note is run through `en_core_web_sm` as it comes, with no changes.

In [10]:
general_rows = []
for ent in doc.ents:
    general_rows.append({
        "Entity Text": ent.text,
        "Label": ent.label_,
        "Start": ent.start_char,
        "End": ent.end_char,
        "Meaning": spacy.explain(ent.label_),
    })

pd.DataFrame(general_rows)

,Entity Text,Label,Start,End,Meaning
0,56-year-old,DATE,2,13,Absolute or relative dates or periods
1,3 days,DATE,77,83,Absolute or relative dates or periods
2,Azithromycin,PERSON,185,197,"People, including fictional"
3,500,CARDINAL,198,201,Numerals that do not fall under another type
4,5 days,DATE,220,226,Absolute or relative dates or periods
5,Paracetamol 650,LAW,231,246,Named documents made into laws.


**Documenting what the general model actually produces.** As the lab sheet
warns, `en_core_web_sm` is trained on news and web text, not clinical text. On
this note it produces generic labels only: it tags dosages as `QUANTITY`,
durations as `DATE`, and it labels the drug **Azithromycin** as `PERSON`. It does
not recognise `pneumonia` as a disease at all.

These are the labels the model really returns and they are reported as they are,
without editing them by hand.

In [11]:
check = nlp("Patient was diagnosed with pneumonia and prescribed Azithromycin 500 mg.")

print("Entities found by the general model:")
for ent in check.ents:
    print(f"  {ent.text:<15} -> {ent.label_}")
print()
print("'pneumonia' recognised:", any(ent.text == "pneumonia" for ent in check.ents))

Entities found by the general model:
  Azithromycin    -> PERSON

'pneumonia' recognised: False


### C2. Adding a clinical entity layer

To get healthcare categories, a rule-based `EntityRuler` holding a small clinical
vocabulary is added to the pipeline. The statistical `ner` component is excluded
so that only clinical entities are returned.

This is the practical stand-in for a clinical-domain model such as scispaCy or
medspaCy, which would be used in a real deployment.

In [12]:
CLINICAL_TERMS = {
    "DISEASE": [
        "pneumonia", "type 2 diabetes mellitus", "diabetes mellitus",
        "myocardial infarction", "meningitis", "urinary tract infection",
        "heart failure", "hypertension", "rheumatoid arthritis",
        "acute gastroenteritis", "pulmonary tuberculosis", "gastritis",
        "dengue fever",
    ],
    "SYMPTOM": [
        "fever", "high-grade fever", "productive cough", "persistent cough",
        "chest pain", "severe chest pain", "increased thirst",
        "frequent urination", "headache", "photophobia", "neck stiffness",
        "burning micturition", "abdominal pain", "lower abdominal pain",
        "breathlessness", "leg swelling", "joint pain", "morning stiffness",
        "loose stools", "vomiting", "weight loss", "epigastric pain",
        "acid reflux", "rash", "wound pain",
    ],
    "MEDICATION": [
        "Azithromycin", "Paracetamol", "Metformin", "Nitrofurantoin",
        "Furosemide", "Methotrexate", "Ondansetron", "Pantoprazole",
        "Ibuprofen",
    ],
    "INVESTIGATION": [
        "chest x-ray", "ECG", "MRI brain", "urine culture", "echocardiography",
        "rheumatoid factor", "stool examination", "sputum examination",
        "upper GI endoscopy", "dengue NS1 antigen", "platelet count",
        "abdominal ultrasound", "blood glucose",
    ],
    "ANATOMY": [
        "chest", "left arm", "brain", "neck", "hands", "abdomen",
    ],
    "PROCEDURE": [
        "angioplasty", "lumbar puncture", "appendectomy",
    ],
    "FREQUENCY": [
        "once daily", "twice daily", "three times daily", "weekly",
        "as required",
    ],
}

CLINICAL_LABELS = list(CLINICAL_TERMS) + ["DOSAGE"]
print("Clinical entity types:", ", ".join(CLINICAL_LABELS))
print("Vocabulary size:", sum(len(v) for v in CLINICAL_TERMS.values()), "terms")

Clinical entity types: DISEASE, SYMPTOM, MEDICATION, INVESTIGATION, ANATOMY, PROCEDURE, FREQUENCY, DOSAGE
Vocabulary size: 74 terms


In [13]:
def build_clinical_nlp():
    # The general 'ner' component is excluded so only clinical entities come back
    clinical = spacy.load("en_core_web_sm", exclude=["ner"])
    ruler = clinical.add_pipe("entity_ruler")

    patterns = []
    for label, phrases in CLINICAL_TERMS.items():
        for phrase in phrases:
            # Build the pattern from the tokenizer so hyphenated terms such as
            # 'X-ray' are matched correctly
            tokens = [{"LOWER": t.text.lower()} for t in clinical.tokenizer(phrase)]
            patterns.append({"label": label, "pattern": tokens})

    # Dosage: any number followed by a unit, e.g. '500 mg'
    for unit in ["mg", "ml", "g", "mcg"]:
        patterns.append({
            "label": "DOSAGE",
            "pattern": [{"LIKE_NUM": True}, {"LOWER": unit}],
        })

    ruler.add_patterns(patterns)
    return clinical


clinical_nlp = build_clinical_nlp()
print("Pipeline components:", clinical_nlp.pipe_names)
print("Patterns loaded:", len(clinical_nlp.get_pipe("entity_ruler").patterns))

Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'entity_ruler']
Patterns loaded: 78


### C3. Extract recognized entities

In [14]:
clinical_doc = clinical_nlp(note)

ent_rows = []
for ent in clinical_doc.ents:
    ent_rows.append({
        "Entity Text": ent.text,
        "Entity Label": ent.label_,
        "Start": ent.start_char,
        "End": ent.end_char,
    })

entities_df = pd.DataFrame(ent_rows)
print("Entities found:", len(entities_df))
entities_df

Entities found: 11


,Entity Text,Entity Label,Start,End
0,fever,SYMPTOM,34,39
1,productive cough,SYMPTOM,41,57
2,chest pain,SYMPTOM,62,72
3,Chest X-ray,INVESTIGATION,85,96
4,pneumonia,DISEASE,147,156
5,Azithromycin,MEDICATION,185,197
6,500 mg,DOSAGE,198,204
7,once daily,FREQUENCY,205,215
8,Paracetamol,MEDICATION,231,242
9,650 mg,DOSAGE,243,249


In [15]:
example_doc = clinical_nlp("Patient was diagnosed with pneumonia and prescribed Azithromycin 500 mg.")

for ent in example_doc.ents:
    print(f"{ent.text:<15} -> {ent.label_}")

pneumonia       -> DISEASE
Azithromycin    -> MEDICATION
500 mg          -> DOSAGE


### C4. Visualize the entities

In [16]:
COLORS = {
    "DISEASE": "#ff9aa2",
    "SYMPTOM": "#ffdac1",
    "MEDICATION": "#b5ead7",
    "DOSAGE": "#c7ceea",
    "INVESTIGATION": "#e2f0cb",
    "ANATOMY": "#f5d5e0",
    "PROCEDURE": "#d4a5a5",
    "FREQUENCY": "#dcd3ff",
}
options = {"ents": CLINICAL_LABELS, "colors": COLORS}

displacy.render(clinical_doc, style="ent", jupyter=True, options=options)

## Part D - Simple Clinical Information Extraction Pipeline

```
Clinical Note -> Preprocessing -> Tokenization -> Chunking -> NER -> Structured Output
```

In [17]:
CATEGORY_NAMES = {
    "DISEASE": "Disease",
    "SYMPTOM": "Symptom",
    "MEDICATION": "Medication",
    "DOSAGE": "Dosage",
    "INVESTIGATION": "Investigation",
    "ANATOMY": "Anatomy",
    "PROCEDURE": "Procedure",
    "FREQUENCY": "Frequency",
}


def extract_clinical_info(text):
    """Run the full pipeline on one clinical note and return the structured output."""
    processed = " ".join(text.split())          # preprocessing: tidy up whitespace
    doc = clinical_nlp(processed)

    return {
        "note": processed,
        "tokens": [t.text for t in doc],
        "chunks": [c.text for c in doc.noun_chunks if is_meaningful(c)],
        "entities": pd.DataFrame(
            [{"Text": e.text, "Category": CATEGORY_NAMES[e.label_], "Label": e.label_}
             for e in doc.ents]
        ),
    }

In [18]:
new_note = ("A patient with diabetes mellitus reported increased thirst and "
            "frequent urination. Metformin 500 mg was continued.")

result = extract_clinical_info(new_note)

print("Tokens:", len(result["tokens"]))
print(" | ".join(result["tokens"][:15]), "...")
print()
print("Noun phrases:")
for c in result["chunks"]:
    print(" -", c)

Tokens: 18
A | patient | with | diabetes | mellitus | reported | increased | thirst | and | frequent | urination | . | Metformin | 500 | mg ...

Noun phrases:
 - diabetes
 - mellitus
 - thirst and frequent urination
 - Metformin
 - 500 mg


In [19]:
result["entities"]

,Text,Category,Label
0,diabetes mellitus,Disease,DISEASE
1,increased thirst,Symptom,SYMPTOM
2,frequent urination,Symptom,SYMPTOM
3,Metformin,Medication,MEDICATION
4,500 mg,Dosage,DOSAGE


### The full output for one note, in the format from the lab sheet

In [20]:
def show_report(text):
    out = extract_clinical_info(text)
    line = "-" * 50

    print("CLINICAL NOTE")
    print(line)
    print(out["note"])
    print()

    print("TOKENS")
    print(line)
    print(" | ".join(out["tokens"]))
    print()

    print("CHUNKS")
    print(line)
    for c in out["chunks"]:
        print(c)
    print()

    print("NAMED ENTITIES")
    print(line)
    for _, row in out["entities"].iterrows():
        print(f"{row['Text']} -> {row['Label']}")
    print()

    print("STRUCTURED CLINICAL INFORMATION")
    print(line)
    for category in out["entities"]["Category"].unique():
        values = out["entities"].loc[out["entities"]["Category"] == category, "Text"]
        print(f"{category}: {', '.join(dict.fromkeys(values))}")


show_report("Patient presented with fever and persistent cough. "
            "Chest X-ray showed pneumonia. Azithromycin 500 mg was prescribed.")

CLINICAL NOTE
--------------------------------------------------
Patient presented with fever and persistent cough. Chest X-ray showed pneumonia. Azithromycin 500 mg was prescribed.

TOKENS
--------------------------------------------------
Patient | presented | with | fever | and | persistent | cough | . | Chest | X | - | ray | showed | pneumonia | . | Azithromycin | 500 | mg | was | prescribed | .

CHUNKS
--------------------------------------------------
fever
persistent cough
Chest X
ray
pneumonia
Azithromycin
500 mg

NAMED ENTITIES
--------------------------------------------------
fever -> SYMPTOM
persistent cough -> SYMPTOM
Chest X-ray -> INVESTIGATION
pneumonia -> DISEASE
Azithromycin -> MEDICATION
500 mg -> DOSAGE

STRUCTURED CLINICAL INFORMATION
--------------------------------------------------
Symptom: fever, persistent cough
Investigation: Chest X-ray
Disease: pneumonia
Medication: Azithromycin
Dosage: 500 mg


### Running the pipeline over the whole dataset

In [21]:
all_rows = []
for _, row in notes_df.iterrows():
    for ent in clinical_nlp(row["note_text"]).ents:
        all_rows.append({
            "Note ID": row["note_id"],
            "Text": ent.text,
            "Category": CATEGORY_NAMES[ent.label_],
            "Label": ent.label_,
        })

all_entities = pd.DataFrame(all_rows)
print("Total entities extracted from", len(notes_df), "notes:", len(all_entities))
print()
print(all_entities["Category"].value_counts().to_string())

Total entities extracted from 12 notes: 81



Category
Symptom          24
Investigation    13
Disease          12
Medication        9
Dosage            9
Frequency         9
Procedure         3
Anatomy           2


In [22]:
all_entities.head(20)

,Note ID,Text,Category,Label
0,1,fever,Symptom,SYMPTOM
1,1,productive cough,Symptom,SYMPTOM
2,1,chest pain,Symptom,SYMPTOM
3,1,Chest X-ray,Investigation,INVESTIGATION
4,1,pneumonia,Disease,DISEASE
5,1,Azithromycin,Medication,MEDICATION
6,1,500 mg,Dosage,DOSAGE
7,1,once daily,Frequency,FREQUENCY
8,1,Paracetamol,Medication,MEDICATION
9,1,650 mg,Dosage,DOSAGE


## Part E - Evaluate the NER Output

Five test notes are annotated by hand with the entities that should be found.
The annotations are written independently of the vocabulary above, so the test
includes terms the pipeline has never seen (`Amoxicillin`, `CT scan`,
`acute appendicitis`).

In [23]:
TEST_SET = [
    ("The patient developed severe abdominal pain and was prescribed Amoxicillin 500 mg.", [
        ("severe abdominal pain", "SYMPTOM"),
        ("Amoxicillin", "MEDICATION"),
        ("500 mg", "DOSAGE"),
    ]),
    ("Patient presented with fever and persistent cough. Chest X-ray showed pneumonia. "
     "Azithromycin 500 mg was prescribed.", [
        ("fever", "SYMPTOM"),
        ("persistent cough", "SYMPTOM"),
        ("Chest X-ray", "INVESTIGATION"),
        ("pneumonia", "DISEASE"),
        ("Azithromycin", "MEDICATION"),
        ("500 mg", "DOSAGE"),
    ]),
    ("A patient with diabetes mellitus reported increased thirst and frequent urination. "
     "Metformin 500 mg was continued.", [
        ("diabetes mellitus", "DISEASE"),
        ("increased thirst", "SYMPTOM"),
        ("frequent urination", "SYMPTOM"),
        ("Metformin", "MEDICATION"),
        ("500 mg", "DOSAGE"),
    ]),
    ("Patient complained of headache and vomiting. CT scan of the brain was advised.", [
        ("headache", "SYMPTOM"),
        ("vomiting", "SYMPTOM"),
        ("CT scan", "INVESTIGATION"),
        ("brain", "ANATOMY"),
    ]),
    ("The patient underwent appendectomy for acute appendicitis. "
     "Ibuprofen 400 mg three times daily was given.", [
        ("appendectomy", "PROCEDURE"),
        ("acute appendicitis", "DISEASE"),
        ("Ibuprofen", "MEDICATION"),
        ("400 mg", "DOSAGE"),
        ("three times daily", "FREQUENCY"),
    ]),
]

print("Test notes:", len(TEST_SET))
print("Annotated entities:", sum(len(gold) for _, gold in TEST_SET))

Test notes: 5
Annotated entities: 23


In [24]:
def normalise(pairs):
    """Compare on lowercased text plus label, so casing does not affect the score."""
    return {(text.lower(), label) for text, label in pairs}


correct = 0
total_predicted = 0
total_actual = 0
comparison = []

for text, gold in TEST_SET:
    predicted = [(e.text, e.label_) for e in clinical_nlp(text).ents]

    gold_set = normalise(gold)
    pred_set = normalise(predicted)
    matched = gold_set & pred_set

    correct += len(matched)
    total_predicted += len(pred_set)
    total_actual += len(gold_set)

    for item in sorted(gold_set - pred_set):
        comparison.append({"Entity": item[0], "Type": item[1], "Result": "missed (false negative)"})
    for item in sorted(pred_set - gold_set):
        comparison.append({"Entity": item[0], "Type": item[1], "Result": "extra (false positive)"})

print("Correctly Extracted Entities:", correct)
print("Total Extracted Entities:   ", total_predicted)
print("Total Actual Entities:      ", total_actual)

Correctly Extracted Entities: 19
Total Extracted Entities:    20
Total Actual Entities:       23


In [25]:
precision = correct / total_predicted
recall = correct / total_actual
f1 = 2 * precision * recall / (precision + recall)

print("EVALUATION")
print("-" * 50)
print(f"Precision : {precision:.3f}")
print(f"Recall    : {recall:.3f}")
print(f"F1-Score  : {f1:.3f}")

EVALUATION
--------------------------------------------------
Precision : 0.950
Recall    : 0.826
F1-Score  : 0.884


### Where the pipeline went wrong

In [26]:
pd.DataFrame(comparison)

,Entity,Type,Result
0,amoxicillin,MEDICATION,missed (false negative)
1,severe abdominal pain,SYMPTOM,missed (false negative)
2,abdominal pain,SYMPTOM,extra (false positive)
3,ct scan,INVESTIGATION,missed (false negative)
4,acute appendicitis,DISEASE,missed (false negative)


The errors come from the rule-based approach: a term that is not in the
vocabulary cannot be found (`Amoxicillin`, `CT scan`, `acute appendicitis`), and
`severe abdominal pain` is matched only as `abdominal pain`, so the boundary is
wrong and it counts as both a miss and a false positive. A trained clinical NER
model would generalise to unseen drug and disease names instead of relying on a
fixed list.

In [27]:
per_label = {}
for text, gold in TEST_SET:
    pred_set = normalise([(e.text, e.label_) for e in clinical_nlp(text).ents])
    gold_set = normalise(gold)

    for _, label in gold_set:
        per_label.setdefault(label, {"actual": 0, "predicted": 0, "correct": 0})
        per_label[label]["actual"] += 1
    for _, label in pred_set:
        per_label.setdefault(label, {"actual": 0, "predicted": 0, "correct": 0})
        per_label[label]["predicted"] += 1
    for _, label in gold_set & pred_set:
        per_label[label]["correct"] += 1

label_rows = []
for label, c in sorted(per_label.items()):
    p = c["correct"] / c["predicted"] if c["predicted"] else 0.0
    r = c["correct"] / c["actual"] if c["actual"] else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    label_rows.append({"Entity Type": label, "Actual": c["actual"],
                       "Predicted": c["predicted"], "Correct": c["correct"],
                       "Precision": round(p, 3), "Recall": round(r, 3),
                       "F1": round(f, 3)})

pd.DataFrame(label_rows)

,Entity Type,Actual,Predicted,Correct,Precision,Recall,F1
0,ANATOMY,1,1,1,1.000,1.000,1.000
1,DISEASE,3,2,2,1.000,0.667,0.800
2,DOSAGE,4,4,4,1.000,1.000,1.000
3,FREQUENCY,1,1,1,1.000,1.000,1.000
4,INVESTIGATION,2,1,1,1.000,0.500,0.667
5,MEDICATION,4,3,3,1.000,0.750,0.857
6,PROCEDURE,1,1,1,1.000,1.000,1.000
7,SYMPTOM,7,7,6,0.857,0.857,0.857


## Part F - Industry-Oriented Application

**Use case: automated discharge summary extraction**

```
Unstructured Discharge Summary
        |
   NLP Pipeline
        |
 Chunking + NER
        |
Structured Patient Information
        |
EHR / Clinical Analytics System
```

The same pipeline applied to a discharge summary turns free text into fields
that can be written into an EHR or searched across a whole archive of notes.

In [28]:
discharge_summary = (
    "The patient was admitted with high-grade fever, productive cough and chest pain. "
    "Chest X-ray showed right lower-lobe consolidation and a diagnosis of pneumonia was made. "
    "Known hypertension was noted. Azithromycin 500 mg once daily was given for 5 days "
    "along with Paracetamol 650 mg as required. The patient improved and was discharged."
)

show_report(discharge_summary)

CLINICAL NOTE
--------------------------------------------------
The patient was admitted with high-grade fever, productive cough and chest pain. Chest X-ray showed right lower-lobe consolidation and a diagnosis of pneumonia was made. Known hypertension was noted. Azithromycin 500 mg once daily was given for 5 days along with Paracetamol 650 mg as required. The patient improved and was discharged.

TOKENS
--------------------------------------------------
The | patient | was | admitted | with | high | - | grade | fever | , | productive | cough | and | chest | pain | . | Chest | X | - | ray | showed | right | lower | - | lobe | consolidation | and | a | diagnosis | of | pneumonia | was | made | . | Known | hypertension | was | noted | . | Azithromycin | 500 | mg | once | daily | was | given | for | 5 | days | along | with | Paracetamol | 650 | mg | as | required | . | The | patient | improved | and | was | discharged | .

CHUNKS
--------------------------------------------------
high-gr

### The same information as structured EHR fields

In [29]:
summary_doc = clinical_nlp(discharge_summary)

ehr_fields = {}
for ent in summary_doc.ents:
    field = CATEGORY_NAMES[ent.label_]
    ehr_fields.setdefault(field, [])
    if ent.text not in ehr_fields[field]:
        ehr_fields[field].append(ent.text)

pd.DataFrame([{"EHR Field": k, "Extracted Value": ", ".join(v)} for k, v in ehr_fields.items()])

,EHR Field,Extracted Value
0,Symptom,"high-grade fever, productive cough, chest pain"
1,Investigation,Chest X-ray
2,Disease,"pneumonia, hypertension"
3,Medication,"Azithromycin, Paracetamol"
4,Dosage,"500 mg, 650 mg"
5,Frequency,"once daily, as required"


In [30]:
short_example = "Patient diagnosed with Type 2 diabetes mellitus. Metformin 500 mg twice daily was prescribed."

for ent in clinical_nlp(short_example).ents:
    print(f"{CATEGORY_NAMES[ent.label_]} -> {ent.text}")

Disease -> Type 2 diabetes mellitus
Medication -> Metformin
Dosage -> 500 mg
Frequency -> twice daily


The pipeline output can support automatic identification of diagnoses,
extraction of medications and dosage, symptom capture, retrieval of investigation
results, record summarization, and search across large collections of clinical
notes.

## Result

The NLP-based clinical text processing system was implemented to perform
tokenization, chunking and Named Entity Recognition on synthetic clinical notes.
The system extracted meaningful phrases and clinically relevant entities from
unstructured healthcare text and presented them in a structured format, and the
entity output was evaluated using precision, recall and F1-score.